# IPSR Unit 5: Uncertain Knowledge and Reasoning

Subject: Introduction to Problem Solving and Reasoning (B.Tech Semester I).

Eleven topics from the Unit V syllabus: uncertainty, basic probability, conditional probability, the joint distribution, Bayes rule, Bayesian reasoning and ranking hypotheses. Plain Python plus a little pandas.

**How to use this notebook:** read a section, run every code cell with Shift+Enter, then try the exercises. Each exercise cell has only comments; write your answer there. Solutions are hidden: click *Show solution* only after you have tried.

---
## 5.1 Objectives

*HTML lab: Unit 5 lab, lesson 1*

By the end of this unit you can:
- explain why an agent cannot rely on true/false logic alone in the real world,
- compute and combine probabilities (sum rule, conditional probability, joint distribution),
- update a belief with **Bayes rule** when new evidence arrives,
- rank competing hypotheses by how probable they are given the evidence.

The key idea: an agent that cannot be certain should still act, by choosing the action with the best **expected utility**.

In [1]:
# Should I carry an umbrella? Probability of rain 30%.
p_rain = 0.30
utility = {("umbrella","rain"): 5, ("umbrella","dry"): -1, ("none","rain"): -10, ("none","dry"): 3}
for action in ("umbrella", "none"):
    eu = p_rain*utility[(action,"rain")] + (1-p_rain)*utility[(action,"dry")]
    print(f"{action:9} expected utility = {eu:.2f}")

umbrella  expected utility = 0.80
none      expected utility = -0.90


**Exercise 1.** Change p_rain to 0.1. Which action is best now?

In [2]:
# Exercise 1: write your code below
# Hint: recompute the loop


<details><summary>Show solution</summary>

```python
p_rain = 0.10
for action in ("umbrella", "none"):
    eu = p_rain*utility[(action,"rain")] + (1-p_rain)*utility[(action,"dry")]
    print(action, round(eu, 2))
# 'none' wins: 0.1*(-10)+0.9*3 = 1.7 versus 0.1*5+0.9*(-1) = -0.4
```

</details>

---
## 5.2 Uncertainty

*HTML lab: Unit 5 lab, lesson 1*

A logical rule such as *toothache implies cavity* is too strong: many toothaches have other causes. With a simulated clinic of 10,000 patients we can see how often the rule fails.

In [3]:
import numpy as np
rng = np.random.default_rng(1)
N = 10000
cavity = rng.random(N) < 0.10
toothache = np.where(cavity, rng.random(N) < 0.80, rng.random(N) < 0.10)
print("patients with toothache:", toothache.sum())
print("of those, with a cavity:", (toothache & cavity).sum())
print("so 'toothache -> cavity' is wrong for", round(100*(toothache & ~cavity).sum()/toothache.sum(), 1), "% of toothache cases")

patients with toothache: 1673
of those, with a cavity: 805
so 'toothache -> cavity' is wrong for 51.9 % of toothache cases


**Exercise 1.** What fraction of cavity patients have NO toothache?

In [4]:
# Exercise 1: write your code below
# Hint: cavity & ~toothache


<details><summary>Show solution</summary>

```python
print(round((cavity & ~toothache).sum() / cavity.sum(), 3))
```

</details>

---
## 5.3 Sources of uncertain knowledge

*HTML lab: Unit 5 lab, lesson 1*

In [5]:
sources = {
 "Laziness":              "too many exceptions to list every rule (a rule for every cause of toothache)",
 "Theoretical ignorance": "the science itself is incomplete (no full theory of the disease)",
 "Practical ignorance":   "we know the rules but cannot measure every fact about this patient",
 "Noisy sensors":         "readings are approximate",
 "Incomplete data":       "some values are missing",
}
for k, v in sources.items():
    print(f"{k:22} {v}")
rng = np.random.default_rng(3)
truth = 36.8
reads = truth + rng.normal(0, 0.4, 5)
print("\nfive noisy thermometer readings:", reads.round(2))
print("their average:", reads.mean().round(2), "(true", truth, ")")

Laziness               too many exceptions to list every rule (a rule for every cause of toothache)
Theoretical ignorance  the science itself is incomplete (no full theory of the disease)
Practical ignorance    we know the rules but cannot measure every fact about this patient
Noisy sensors          readings are approximate
Incomplete data        some values are missing

five noisy thermometer readings: [37.62 35.78 36.97 36.57 36.62]
their average: 36.71 (true 36.8 )


**Exercise 1.** Which source of uncertainty is a blurred X-ray photo? Answer in a comment.

In [6]:
# Exercise 1: write your code below
# Hint: think: sensors or science?


<details><summary>Show solution</summary>

```python
# Noisy sensors (and some practical ignorance: we cannot see everything inside the body).
```

</details>

---
## 5.4 Probabilistic reasoning

*HTML lab: Unit 5 lab, lesson 2*

**Probabilistic reasoning** keeps a *degree of belief* between 0 and 1 for each statement instead of just true or false. An agent then picks the action with maximum expected utility. Classic example: leaving for the airport.

In [7]:
plans = {                     # (probability of catching the flight, comfort utility)
  "Leave 90 min early":  (0.95, -3),
  "Leave 120 min early": (0.98, -6),
  "Leave 180 min early": (0.999, -12),
}
miss_cost = -100
for name, (p, comfort) in plans.items():
    eu = p*0 + (1-p)*miss_cost + comfort
    print(f"{name:21} P(catch)={p:<6} expected utility={eu:7.2f}")

Leave 90 min early    P(catch)=0.95   expected utility=  -8.00
Leave 120 min early   P(catch)=0.98   expected utility=  -8.00
Leave 180 min early   P(catch)=0.999  expected utility= -12.10


**Exercise 1.** If missing the flight costs -1000 instead of -100, which plan is best?

In [8]:
# Exercise 1: write your code below
# Hint: recompute with miss_cost = -1000


<details><summary>Show solution</summary>

```python
miss_cost = -1000
best = max(plans, key=lambda k: (1-plans[k][0])*miss_cost + plans[k][1])
print(best)
```

</details>

---
## 5.5 Basic probability theory

*HTML lab: Unit 5 lab, lesson 2*

Rules: 0 <= P(A) <= 1; P(certain) = 1; **complement** P(not A) = 1 - P(A); **addition** P(A or B) = P(A) + P(B) - P(A and B). For equally likely outcomes P(A) = favourable / total.

In [9]:
from fractions import Fraction
from itertools import product
space = list(product(range(1, 7), repeat=2))          # two dice, 36 outcomes
def P(event): return Fraction(sum(1 for o in space if event(o)), len(space))
sum7      = lambda o: o[0] + o[1] == 7
double    = lambda o: o[0] == o[1]
print("P(sum is 7)        =", P(sum7))
print("P(double)          =", P(double))
print("P(not double)      =", 1 - P(double))
both = P(lambda o: sum7(o) and double(o))
print("P(sum 7 or double) =", P(sum7) + P(double) - both)
rng = np.random.default_rng(0)
for n_ in (100, 10000, 1000000):
    rolls = rng.integers(1, 7, (n_, 2)).sum(axis=1)
    print(n_, "rolls: share with sum 7 =", round((rolls == 7).mean(), 4), "(true", round(6/36, 4), ")")

P(sum is 7)        = 1/6
P(double)          = 1/6
P(not double)      = 5/6
P(sum 7 or double) = 1/3
100 rolls: share with sum 7 = 0.16 (true 0.1667 )
10000 rolls: share with sum 7 = 0.1696 (true 0.1667 )
1000000 rolls: share with sum 7 = 0.1668 (true 0.1667 )


**Exercise 1.** P(sum is at least 10)?

In [10]:
# Exercise 1: write your code below
# Hint: count outcomes


<details><summary>Show solution</summary>

```python
print(P(lambda o: o[0] + o[1] >= 10))
```

</details>

---
## 5.6 Conditional probability

*HTML lab: Unit 5 lab, lesson 3*

**P(A | B)** = P(A and B) / P(B): the probability of A once we *know* B. Knowing B shrinks the sample space to B only.

In [11]:
import pandas as pd
t = pd.DataFrame({"Passed": [420, 90], "Failed": [60, 130]}, index=["Attended >= 75%", "Attended < 75%"])
print(t, "\n")
total = t.values.sum()
p_pass = t["Passed"].sum()/total
p_att  = t.loc["Attended >= 75%"].sum()/total
p_both = t.loc["Attended >= 75%", "Passed"]/total
print("P(pass)                =", round(p_pass, 3))
print("P(pass | attended)     =", round(p_both/p_att, 3))
print("P(pass | not attended) =", round(t.loc["Attended < 75%","Passed"]/t.loc["Attended < 75%"].sum(), 3))
print("independent? P(pass|attended) == P(pass)?", abs(p_both/p_att - p_pass) < 0.01)

                 Passed  Failed
Attended >= 75%     420      60
Attended < 75%       90     130 

P(pass)                = 0.729
P(pass | attended)     = 0.875
P(pass | not attended) = 0.409
independent? P(pass|attended) == P(pass)? False


**Exercise 1.** Compute P(attended | failed).

In [12]:
# Exercise 1: write your code below
# Hint: condition on the Failed column


<details><summary>Show solution</summary>

```python
print(round(t.loc["Attended >= 75%", "Failed"] / t["Failed"].sum(), 3))
```

</details>

---
## 5.7 Bayesian rule

*HTML lab: Unit 5 lab, lesson 5*

**Bayes rule:** P(H | E) = P(E | H) x P(H) / P(E), where P(E) = P(E|H) P(H) + P(E|not H) P(not H).

It turns *how likely the evidence is if the hypothesis is true* (easy to know) into *how likely the hypothesis is given the evidence* (what we want).

In [13]:
def bayes(prior, p_e_given_h, p_e_given_not_h):
    p_e = p_e_given_h*prior + p_e_given_not_h*(1 - prior)
    return p_e_given_h*prior / p_e
# A disease affects 1% of people. The test finds 90% of sick people and wrongly flags 5% of healthy people.
post = bayes(0.01, 0.90, 0.05)
print("P(disease | positive test) =", round(post, 3))
print("\nNatural frequencies, 10,000 people:")
sick = 100; healthy = 9900
tp = 0.90*sick; fp = 0.05*healthy
print(f"  sick and positive   : {tp:.0f}")
print(f"  healthy but positive: {fp:.0f}")
print(f"  of {tp+fp:.0f} positives only {tp:.0f} are sick = {tp/(tp+fp):.3f}")

P(disease | positive test) = 0.154

Natural frequencies, 10,000 people:
  sick and positive   : 90
  healthy but positive: 495
  of 585 positives only 90 are sick = 0.154


**Exercise 1.** What is the posterior if the disease affects 10% of people (same test)?

In [14]:
# Exercise 1: write your code below
# Hint: change the prior


<details><summary>Show solution</summary>

```python
print(round(bayes(0.10, 0.90, 0.05), 3))
```

</details>

---
## 5.8 The joint probability

*HTML lab: Unit 5 lab, lesson 4*

The **full joint distribution** lists the probability of every combination of variables. Everything else (marginals, conditionals) can be computed from it.

In [15]:
import itertools
joint = {  # (toothache, catch, cavity) -> probability
 (1,1,1):0.108, (1,1,0):0.016, (1,0,1):0.012, (1,0,0):0.064,
 (0,1,1):0.072, (0,1,0):0.144, (0,0,1):0.008, (0,0,0):0.576}
print("sum of all entries:", round(sum(joint.values()), 3))
def prob(f): return sum(p for k, p in joint.items() if f(*k))
print("P(toothache)             =", round(prob(lambda t,c,v: t), 3))
print("P(cavity)                =", round(prob(lambda t,c,v: v), 3))
print("P(cavity | toothache)    =", round(prob(lambda t,c,v: v and t)/prob(lambda t,c,v: t), 3))
print("P(no cavity | toothache) =", round(prob(lambda t,c,v: (not v) and t)/prob(lambda t,c,v: t), 3))
print("P(toothache or cavity)   =", round(prob(lambda t,c,v: t or v), 3))

sum of all entries: 1.0
P(toothache)             = 0.2
P(cavity)                = 0.2
P(cavity | toothache)    = 0.6
P(no cavity | toothache) = 0.4
P(toothache or cavity)   = 0.28


**Exercise 1.** Compute P(cavity | toothache and catch).

In [16]:
# Exercise 1: write your code below
# Hint: condition on both


<details><summary>Show solution</summary>

```python
print(round(prob(lambda t,c,v: v and t and c) / prob(lambda t,c,v: t and c), 3))
```

</details>

---
## 5.9 Bayesian reasoning

*HTML lab: Unit 5 lab, lesson 5*

**Bayesian reasoning** means updating belief step by step: today's posterior becomes tomorrow's prior when new evidence arrives. Below, a tiny spam filter reads words one at a time.

In [17]:
p_word = {"offer": (0.30, 0.02), "free": (0.40, 0.03), "meeting": (0.02, 0.25), "report": (0.03, 0.20)}  # (P(word|spam), P(word|ham))
belief = 0.20                                    # prior P(spam)
print(f"start: P(spam) = {belief:.3f}")
for w in ["offer", "free", "meeting"]:
    ps, ph = p_word[w]
    belief = ps*belief / (ps*belief + ph*(1 - belief))
    print(f"after seeing '{w}': P(spam) = {belief:.3f}")

start: P(spam) = 0.200
after seeing 'offer': P(spam) = 0.789
after seeing 'free': P(spam) = 0.980
after seeing 'meeting': P(spam) = 0.800


**Exercise 1.** Start again from 0.20 and read 'report' then 'meeting'. What is P(spam)?

In [18]:
# Exercise 1: write your code below
# Hint: same loop, different words


<details><summary>Show solution</summary>

```python
b = 0.20
for w in ["report", "meeting"]:
    ps, ph = p_word[w]
    b = ps*b / (ps*b + ph*(1 - b))
print(round(b, 4))
```

</details>

---
## 5.10 Ranking potentially true hypotheses

*HTML lab: Unit 5 lab, lesson 6*

With several possible explanations, compute P(H | evidence) for each, then **rank** them. Normalise so they sum to 1. The top one is the *maximum a posteriori* (MAP) hypothesis. (This simple version assumes each symptom is independent given the disease.)

In [19]:
priors = {"Flu": 0.10, "Cold": 0.30, "Allergy": 0.20, "Healthy": 0.40}
lik = {  # P(symptom | hypothesis)
 "fever":    {"Flu":0.90, "Cold":0.20, "Allergy":0.02, "Healthy":0.01},
 "sneezing": {"Flu":0.50, "Cold":0.80, "Allergy":0.90, "Healthy":0.05},
 "itchy":    {"Flu":0.05, "Cold":0.05, "Allergy":0.80, "Healthy":0.02},
}
def rank(evidence):
    raw = {h: priors[h] for h in priors}
    for e in evidence:
        for h in raw: raw[h] *= lik[e][h]
    z = sum(raw.values())
    return sorted(((h, p/z) for h, p in raw.items()), key=lambda x: -x[1])
for ev in (["fever"], ["fever", "sneezing"], ["sneezing", "itchy"]):
    print(ev)
    for h, p in rank(ev): print(f"   {h:8} {p:.3f}")

['fever']
   Flu      0.570
   Cold     0.380
   Allergy  0.025
   Healthy  0.025
['fever', 'sneezing']
   Cold     0.496
   Flu      0.465
   Allergy  0.037
   Healthy  0.002
['sneezing', 'itchy']
   Allergy  0.906
   Cold     0.076
   Flu      0.016
   Healthy  0.003


**Exercise 1.** Which hypothesis is most probable for ['sneezing'] alone?

In [20]:
# Exercise 1: write your code below
# Hint: call rank


<details><summary>Show solution</summary>

```python
print(rank(["sneezing"])[0])
```

</details>

---
## 5.11 The prior and conditional probabilities

*HTML lab: Unit 5 lab, lesson 4*

The **prior** P(H) is belief before the evidence; the **conditional** P(E|H) is how the evidence behaves under each hypothesis; the **posterior** combines them. A rare condition stays unlikely even after a positive test: the prior matters (the *base-rate effect*).

In [21]:
import pandas as pd
rows = []
for prior in (0.001, 0.01, 0.05, 0.10, 0.30, 0.50):
    rows.append({"prior P(disease)": prior, "posterior after + test": round(bayes(prior, 0.90, 0.05), 3)})
print(pd.DataFrame(rows).to_string(index=False))
# estimating a prior and conditionals from data
rng = np.random.default_rng(5)
df = pd.DataFrame({"spam": rng.random(500) < 0.25})
df["has_free"] = np.where(df.spam, rng.random(500) < 0.4, rng.random(500) < 0.03)
print("\nprior P(spam)         =", round(df.spam.mean(), 3))
print("P(free | spam)        =", round(df[df.spam].has_free.mean(), 3))
print("P(free | not spam)    =", round(df[~df.spam].has_free.mean(), 3))

 prior P(disease)  posterior after + test
            0.001                   0.018
            0.010                   0.154
            0.050                   0.486
            0.100                   0.667
            0.300                   0.885
            0.500                   0.947

prior P(spam)         = 0.276
P(free | spam)        = 0.435
P(free | not spam)    = 0.028


**Exercise 1.** At which prior does a positive test first give a posterior above 50%?

In [22]:
# Exercise 1: write your code below
# Hint: scan the table


<details><summary>Show solution</summary>

```python
for p in (0.01, 0.05, 0.10, 0.30, 0.50):
    if bayes(p, 0.90, 0.05) > 0.5:
        print(p); break
```

</details>

**Where to read more:** Russell and Norvig, AIMA chapters 12 and 13; Any Bayes rule explainer using natural frequencies